# Spatial Transcriptomics: A First Look

**REQUIRED DAY 2** (time permitting — see [the Day 2 README](../README.md) for how this fits the schedule)

## What spatial adds

Everything so far today has been **dissociated** single-cell data: cells were removed from tissue, so you know each cell's transcriptome but not where it physically sat relative to other cells. Spatial transcriptomics keeps (or approximates) that physical location — each measurement comes with x/y coordinates on a tissue section, sometimes alongside a matching histology image.

This unlocks questions dissociated scRNA-seq structurally cannot answer: which cell types are physically adjacent to each other, whether a gene's expression varies across a tissue region, and whether a group of cells forms a spatially coherent structure (a tumor boundary, a follicle, a cortical layer) rather than just a transcriptional cluster.

## The tool: squidpy

[squidpy](https://squidpy.readthedocs.io/) is the scverse ecosystem's spatial analysis tool — it builds directly on `scanpy`/`AnnData`, so a spatial `AnnData` object looks like everything you've worked with today, plus a `.obsm["spatial"]` array of coordinates.

## Load a real spatial dataset

Today's data is mouse brain tissue on 10x Visium (a "spot"-based technology — each measurement is a small tissue spot covering several cells, not a single dissociated cell), bundled with squidpy as a standard teaching dataset. Look up [`sq.datasets.visium_hne_adata`](https://squidpy.readthedocs.io/en/stable/api/squidpy.datasets.visium_hne_adata.html) and load it.

This normally downloads (~330MB) from the internet the first time it's called — already fetched once into today's shared data folder instead, so point squidpy's `path` argument at that cached file rather than the default location:

In [ ]:
import scanpy as sc
import squidpy as sq
import warnings
warnings.filterwarnings("ignore")

SHARED_CACHE = "/tscc/nfs/home/juf009/day2_shared_data/extra_datasets/visium_hne.h5ad"

## Fill in: adata_spatial = sq.datasets.visium_hne_adata(path=SHARED_CACHE)

adata_spatial


## Explore the object — what's new compared to today's other AnnData objects?

- Look at `adata_spatial.obsm["spatial"]` — what shape is it, and what do you think the two columns represent?
- Look at `adata_spatial.uns["spatial"]` — this is where squidpy stores the actual histology image alongside scale factors that map spot coordinates onto image pixels.
- Everything else (`.X`, `.obs`, `.var`) should look completely familiar — that's the point: spatial data is ordinary AnnData plus coordinates and an image, not a different data structure.

In [ ]:
# adata_spatial.obsm["spatial"].shape



## The one new plot: expression on top of tissue

Instead of a UMAP, spatial data lets you plot gene expression (or clusters) directly on the tissue image, at each spot's real physical location. Look up [`sc.pl.spatial`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pl.spatial.html) and plot `total_counts` on the tissue image (`img_key="hires"`).

In [ ]:
## Fill in: sc.pl.spatial(adata_spatial, color="total_counts", img_key="hires")



## QC, normalization, and clustering — same steps as this morning

Nothing about QC or clustering is spatial-specific — you already know every one of these functions from [05_loading_data_and_qc.ipynb](05_loading_data_and_qc.ipynb) through [07_dimensionality_reduction_and_clustering.ipynb](07_dimensionality_reduction_and_clustering.ipynb). Run, in order:

- `sc.pp.filter_cells(adata_spatial, min_counts=500)`
- `sc.pp.normalize_total(adata_spatial, inplace=True)`
- `sc.pp.log1p(adata_spatial)`
- `sc.pp.highly_variable_genes(adata_spatial, flavor="seurat", n_top_genes=2000)`
- `sc.pp.pca(adata_spatial)`
- `sc.pp.neighbors(adata_spatial)`
- `sc.tl.leiden(adata_spatial, resolution=1.0)`
- `sc.tl.umap(adata_spatial)`

In [ ]:
## Fill in the QC/clustering pipeline listed above, then check both plots below




In [ ]:
sc.pl.umap(adata_spatial, color="leiden")
sc.pl.spatial(adata_spatial, color="leiden", img_key="hires")


Now look at both plots side by side. The UMAP tells you which spots are transcriptionally similar. The spatial plot tells you whether those same clusters also form **spatially coherent regions** on the actual tissue — which, for a structure like brain tissue with real anatomical layers, they very much should. A cluster that's scattered randomly across the tissue despite being a clean UMAP cluster is worth a second look.

## Neighborhood enrichment: which cell types/clusters sit next to each other

This is the one genuinely spatial-specific analysis in this notebook: instead of asking "which cells look transcriptionally similar," ask "which clusters are physically next to each other more (or less) often than you'd expect by chance." Build a spatial neighbor graph (based on physical proximity, not gene expression) with [`sq.gr.spatial_neighbors`](https://squidpy.readthedocs.io/en/stable/api/squidpy.gr.spatial_neighbors.html), then run [`sq.gr.nhood_enrichment`](https://squidpy.readthedocs.io/en/stable/api/squidpy.gr.nhood_enrichment.html) (`cluster_key="leiden"`) and plot it with `sq.pl.nhood_enrichment`.

In [ ]:
## Fill in:
## sq.gr.spatial_neighbors(adata_spatial)
## sq.gr.nhood_enrichment(adata_spatial, cluster_key="leiden")
## sq.pl.nhood_enrichment(adata_spatial, cluster_key="leiden")




Bright squares off the diagonal mean two clusters are found next to each other far more than chance would predict — a real spatial relationship, not just "these two clusters both exist somewhere in this tissue." This is the same underlying question as PROJECT.md-style questions about tumor microenvironments: not "which cell types are present," but "which cell types are organized near each other, and does that organization mean anything biologically."

## What's beyond today

Following [sc-best-practices.org's Spatial Omics chapters](https://www.sc-best-practices.org/spatial/neighborhood.html), a few things this notebook didn't cover, worth knowing exist:

- **Spatial domains** — identifying spatially coherent regions by combining the expression neighbor graph with the physical proximity graph, rather than clustering on expression alone.
- **Spatially variable genes** — genes whose expression follows a spatial pattern, not just a cluster pattern.
- **Spatial deconvolution** — many spatial technologies measure a "spot" containing several cells at once, not one cell (exactly like today's Visium data); deconvolution methods (Cell2location, SpatialDWLS, RCTD) estimate the cell-type mixture within each spot using a single-cell reference — which is exactly the kind of annotated reference you built in [08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb).

## Further reading

- [squidpy documentation](https://squidpy.readthedocs.io/)
- [Single-cell best practices — Spatial Omics](https://www.sc-best-practices.org/spatial/neighborhood.html)
- [scanpy: spatial data tutorial](https://scanpy-tutorials.readthedocs.io/en/latest/spatial/basic-analysis.html)
- [Seurat's spatial vignette](https://satijalab.org/seurat/articles/spatial_vignette) — the R/Seurat equivalent workflow (`SCTransform`, `RunPCA`, `FindClusters`, `SpatialDimPlot`).